# CHO WGS 全量分析：物种/株系确认 + Transgene 检测

## 已有结果（5M reads 抽样）

| 项目 | 结果 |
|------|------|
| 比对率 | 99.85% → 确认为中国仓鼠 (*Cricetulus griseus*) |
| DHFR 状态 | 深度比值正常 → DHFR 完整 |
| 初步株系判断 | CHO-K1 或 CHO-S（全量数据可精确确认）|
| 参考基因组 | GCF_003668045.1 CriGri-PICR |
| 原始数据量 | R1: 17 GB, R2: 16 GB（约 230M read pairs，估算 ~30x 覆盖度）|

---

## 原方案评判

### 正确之处 ✅
- 核心算法"宿主减除法"方向正确，是业界标准做法
- 正确警告不能对全基因组原始数据从头组装（需 500 GB+ 内存）
- 工具选择合理：MEGAHIT 省内存，NCBI remote BLAST 省磁盘
- 资源估算大体正确

### 需要修正或补充 ❌

1. **缺少 Task 1 代码**：原方案仅描述 transgene 检测思路，未提供全量 fastp + 全量比对的可执行流程。

2. **Transgene 检测必须基于全量 BAM**：5M 抽样中 unmapped reads 约 10K 条，组装效果差；全量数据有约 345K unmapped reads（按 0.15% 未比对率），灵敏度提升 ~34 倍。

3. **Unmapped reads 提取逻辑不完整**：
   - `-f 4` 只提取"自身未比对"的 read，但无法保证配对完整性
   - 正确做法：分两类提取——`-f 12`（双端均未比对）→ PE 输入；`-f 4 -F 8`（一端未比对）→ SE 补充输入
   - BAM 必须先 name-sort，`samtools fastq` 才能正确还原配对关系

4. **缺少 contig 尺寸过滤**：客户要求忽略太小片段，"几百 bp"的自然下限即 200 bp，应设 `--min-contig-len 200`（MEGAHIT 默认值），BLAST 前同样过滤 ≥200 bp。

5. **全量比对效率优化**：推荐 `fastp --stdout | bwa mem -p | samtools sort` 三段管道，省去写出 ~30 GB 中间 FASTQ 文件的时间和磁盘。

## 执行计划

| 任务 | 内容 | 估计耗时 | 备注 |
|------|------|---------|------|
| Task 1A | fastp + BWA + samtools 三段管道 | 6–8 小时 | 必须在 tmux 中运行 |
| Task 1B | DHFR 株系分析（全量 BAM） | < 30 分钟 | 1A 完成后运行 |
| Task 2A | 提取 unmapped/discordant reads | ~1 小时 | 基于全量 BAM |
| Task 2B | MEGAHIT 从头组装 | 30–60 分钟 | 2A 完成后运行 |
| Task 2C | 过滤 contigs + BLAST | 10–30 分钟 | 依赖网络速度 |

**磁盘预算**：全量 BAM ~100–120 GB；unmapped reads ~1–3 GB；组装结果 <1 GB；服务器现有 700 GB 可用，充足。

## Task 1: 全量 fastp QC + BWA 比对

### 策略：三段管道，零中间 FASTQ 文件

```
fastp --stdout  →  bwa mem -p  →  samtools sort  →  wt1_cho_full.bam
```

**为什么三段管道是并行的**：Unix pipe `|` 会同时启动所有进程——fastp 写出一批 reads 时，bwa 立刻从 stdin 读入并开始比对，bwa 产出 SAM 行时 samtools sort 立刻接收。三者同时运行，因此线程数相加，必须控制总和不超过限额。

**线程分配（合计 56，保留 8 个给系统和网络控制）**：

| 进程 | 参数 | 线程数 | 说明 |
|------|------|-------|------|
| fastp | `-w 8` | 8 | I/O 瓶颈，再多无益 |
| bwa mem | `-t 40` | 40 | 计算密集，分配最多 |
| samtools sort | `-@ 8` | 8 | 排序缓冲 |
| **合计** | | **56** | 保留 8 线程给系统 |

In [ ]:
# ============================================================
# Task 1A: 启动全量比对管道（tmux 后台运行，预计 6-8 小时）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
R1="/home/gao/Dropbox/Keqiang/23TF7FLT4_3_0469165296_wt1_S8_L003_R1_001.fastq.gz"
R2="/home/gao/Dropbox/Keqiang/23TF7FLT4_3_0469165296_wt1_S8_L003_R2_001.fastq.gz"
CHO_REF="/Work_bio/references/Cricetulus_griseus/CriGri-PICR/ncbi_refseq/GCF_003668045.1_CriGri-PICR_genomic.fna"

mkdir -p ${PROJDIR}/{qc,align,results,transgene}

# 线程合计 56（fastp 8 + bwa 40 + samtools sort 8），保留 8 给系统
tmux new-session -d -s cho_full_align "
set -euo pipefail
LOG=${PROJDIR}/align/full_pipeline.log
echo '[START] \$(date)' | tee \${LOG}

fastp \
  -i ${R1} -I ${R2} \
  -j ${PROJDIR}/qc/wt1_full_fastp.json \
  -h ${PROJDIR}/qc/wt1_full_fastp.html \
  -w 8 \
  --stdout \
  2>>\${LOG} | \
bwa mem -t 40 -p \
  -R '@RG\tID:wt1\tSM:wt1\tPL:ILLUMINA' \
  ${CHO_REF} - \
  2>>\${LOG} | \
samtools sort -@ 8 -m 4G \
  -o ${PROJDIR}/align/wt1_cho_full.bam -

samtools index -@ 8 ${PROJDIR}/align/wt1_cho_full.bam
echo '[DONE] \$(date)' | tee -a \${LOG}
"

echo "Job launched in tmux session 'cho_full_align'"
echo "Monitor : tmux capture-pane -t cho_full_align -p"
echo "Log     : tail -f ${PROJDIR}/align/full_pipeline.log"

In [ ]:
# Task 1A 进度监控（随时重新运行此 cell）
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"

echo "=== tmux 最新输出 ==="
tmux capture-pane -t cho_full_align -p 2>/dev/null | tail -15 || echo "(tmux session 不存在或已结束)"

echo ""
echo "=== 日志最后 10 行 ==="
tail -10 ${PROJDIR}/align/full_pipeline.log 2>/dev/null || echo "(日志尚未生成)"

echo ""
echo "=== BAM 文件状态 ==="
ls -lh ${PROJDIR}/align/wt1_cho_full.bam 2>/dev/null || echo "(BAM 尚未生成)"

In [ ]:
# ============================================================
# Task 1B: 全量比对率 + DHFR 株系分析（Task 1A 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
CHO_GFF="/Work_bio/references/Cricetulus_griseus/CriGri-PICR/ncbi_refseq/GCF_003668045.1_CriGri-PICR_genomic.gff.gz"
BAM="${PROJDIR}/align/wt1_cho_full.bam"

echo "=== 全量比对统计 ==="
samtools flagstat ${BAM} | tee ${PROJDIR}/results/cho_full_flagstat.txt

echo ""
echo "=== DHFR 株系分析（全量深度）==="
DHFR_CHR=$(awk '{print $1}' ${PROJDIR}/results/dhfr_gff.txt | head -1)
DHFR_START=$(awk '{print $4}' ${PROJDIR}/results/dhfr_gff.txt | head -1)
DHFR_END=$(awk '{print $5}' ${PROJDIR}/results/dhfr_gff.txt | head -1)

FLANK_START=$((DHFR_START - 500000))
FLANK_END=$((DHFR_END + 500000))
[ ${FLANK_START} -lt 1 ] && FLANK_START=1

DHFR_COV=$(samtools depth -r ${DHFR_CHR}:${DHFR_START}-${DHFR_END} ${BAM} | \
  awk '{s+=$3; n++} END{printf "%.2f", (n>0)?s/n:0}')
FLANK_COV=$(samtools depth -r ${DHFR_CHR}:${FLANK_START}-${FLANK_END} ${BAM} | \
  awk '{s+=$3; n++} END{printf "%.2f", (n>0)?s/n:0}')
DHFR_BASES=$(samtools depth -r ${DHFR_CHR}:${DHFR_START}-${DHFR_END} ${BAM} | wc -l)
DHFR_LEN=$((DHFR_END - DHFR_START))

python3 - <<EOF
dhfr_cov   = float("${DHFR_COV}")
flank_cov  = float("${FLANK_COV}")
dhfr_bases = int("${DHFR_BASES}")
dhfr_len   = int("${DHFR_LEN}")
ratio      = dhfr_cov / max(flank_cov, 0.001)
pct        = dhfr_bases / dhfr_len * 100 if dhfr_len > 0 else 0

print(f"  DHFR 区域深度         : {dhfr_cov:.1f}x")
print(f"  侧翼 ±500kb 深度      : {flank_cov:.1f}x")
print(f"  DHFR 碱基覆盖率       : {pct:.1f}%")
print(f"  深度比值 DHFR/侧翼    : {ratio:.3f}")
print()
if dhfr_cov < 1 and pct < 5:
    verdict = "DHFR 纯合缺失 → CHO-DG44"
elif ratio < 0.3:
    verdict = "DHFR 半合缺失 → CHO-DXB11"
elif ratio > 0.7:
    verdict = f"DHFR 完整 → CHO-K1 或 CHO-S（全量深度 {flank_cov:.0f}x，可信度高）"
else:
    verdict = "结果模糊，请检查 DHFR 坐标是否正确"
print(f"  判断: {verdict}")
EOF

# MultiQC
multiqc ${PROJDIR}/qc/ -o ${PROJDIR}/qc/multiqc_full --force -q
echo ""
echo "MultiQC 全量报告: ${PROJDIR}/qc/multiqc_full/multiqc_report.html"

## Task 2: Transgene（外源基因）检测

### 策略

```
全量 BAM → 提取非宿主 reads → MEGAHIT 从头组装 → 过滤 ≥200 bp → BLAST 鉴定
```

### 两类目标 reads

| 类型 | samtools 参数 | 含义 | 用途 |
|------|--------------|------|------|
| 双端均未比对 | `-f 12 -F 256 -F 2048` | Transgene 内部序列 | PE 输入 MEGAHIT `-1/-2` |
| 仅一端未比对 | `-f 4 -F 8 -F 256 -F 2048` | Transgene/宿主边界 | SE 补充输入 MEGAHIT `-r` |

**注意**：使用 `-F 256 -F 2048` 排除 secondary 和 supplementary alignment，避免重复计数。

### 为什么必须用全量 BAM（而不是 5M 抽样 BAM）

| | 5M 抽样 | 全量 (~230M reads) |
|--|---------|-------------------|
| 估算 unmapped reads | ~10K | ~345K |
| 组装覆盖度 | 极低，难以出 contig | 足以组装几百 bp 以上片段 |
| 灵敏度 | 低 | 高（提升 ~34 倍）|

### MEGAHIT 可用性检查
如果 `regular_bioinfo` 环境没有 megahit，可用 `mag_biobakery` 环境（已确认有 blastn）。

In [ ]:
# ============================================================
# Task 2A: 提取 unmapped reads（基于全量 BAM，约 1 小时）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
BAM="${PROJDIR}/align/wt1_cho_full.bam"
TRANS="${PROJDIR}/transgene"
mkdir -p ${TRANS}

tmux new-session -d -s cho_unmapped "
set -euo pipefail
LOG=${TRANS}/extraction.log
echo '[START unmapped extraction] \$(date)' | tee \${LOG}

# --- 类型 1: 双端均未比对 → 还原为配对 FASTQ ---
echo '[1/2] Extracting both-unmapped pairs...' | tee -a \${LOG}
samtools view -b -f 12 -F 256 -F 2048 ${BAM} | \
  samtools sort -n -@ 8 -m 2G | \
  samtools fastq \
    -1 ${TRANS}/unmapped_R1.fq.gz \
    -2 ${TRANS}/unmapped_R2.fq.gz \
    -0 /dev/null -s /dev/null -
PE_CNT=\$(zcat ${TRANS}/unmapped_R1.fq.gz | awk 'NR%4==1' | wc -l)
echo \"  PE pairs: \${PE_CNT}\" | tee -a \${LOG}

# --- 类型 2: 仅一端未比对（transgene/宿主边界 reads）→ 单端 FASTQ ---
echo '[2/2] Extracting singleton unmapped reads...' | tee -a \${LOG}
samtools view -b -f 4 -F 8 -F 256 -F 2048 ${BAM} | \
  samtools fastq - | gzip > ${TRANS}/unmapped_singleton.fq.gz
SE_CNT=\$(zcat ${TRANS}/unmapped_singleton.fq.gz | awk 'NR%4==1' | wc -l)
echo \"  SE reads: \${SE_CNT}\" | tee -a \${LOG}

echo '[DONE] \$(date)' | tee -a \${LOG}
"

echo "Job launched in tmux session 'cho_unmapped'"
echo "Monitor : tmux capture-pane -t cho_unmapped -p"
echo "Log     : tail -f ${TRANS}/extraction.log"

In [ ]:
# ============================================================
# Task 2B: MEGAHIT 从头组装（Task 2A 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
TRANS="${PROJDIR}/transgene"

# 检查 megahit 所在环境
MEGAHIT_BIN=$(conda run -n mag_biobakery which megahit 2>/dev/null || which megahit 2>/dev/null || echo "")
if [ -z "${MEGAHIT_BIN}" ]; then
    echo "ERROR: megahit not found. Install with: mamba install -n regular_bioinfo -c bioconda megahit"
    exit 1
fi
echo "megahit found: ${MEGAHIT_BIN}"

# 输入文件统计
echo "PE pairs : $(zcat ${TRANS}/unmapped_R1.fq.gz | awk 'NR%4==1' | wc -l)"
echo "SE reads : $(zcat ${TRANS}/unmapped_singleton.fq.gz | awk 'NR%4==1' | wc -l)"

# MEGAHIT 不允许输出目录已存在
[ -d ${TRANS}/megahit_out ] && rm -rf ${TRANS}/megahit_out

tmux new-session -d -s cho_megahit "
conda run -n mag_biobakery megahit \
  -1 ${TRANS}/unmapped_R1.fq.gz \
  -2 ${TRANS}/unmapped_R2.fq.gz \
  -r ${TRANS}/unmapped_singleton.fq.gz \
  -o ${TRANS}/megahit_out \
  --min-contig-len 200 \
  -t 32 \
  -m 0.5 \
  2>&1 | tee ${TRANS}/megahit.log
echo '[MEGAHIT DONE] \$(date)' | tee -a ${TRANS}/megahit.log
"

echo "MEGAHIT launched in tmux 'cho_megahit'"
echo "Monitor : tmux capture-pane -t cho_megahit -p"
echo "Log     : tail -f ${TRANS}/megahit.log"

In [ ]:
# ============================================================
# Task 2C: Contig 过滤 + BLASTn 鉴定（Task 2B 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
TRANS="${PROJDIR}/transgene"
CONTIGS="${TRANS}/megahit_out/final.contigs.fa"

# --- Contig 统计 ---
echo "=== 组装结果统计 ==="
awk '/^>/{next} {print length($0)}' ${CONTIGS} | sort -rn | \
  awk 'BEGIN{n=0; s=0}
    {n++; s+=$1; if(n==1)max=$1}
    $1>=1000{c1000++} $1>=500{c500++} $1>=200{c200++}
    END{
      print "Total contigs   : " n
      print "Total bases     : " s
      print "Longest contig  : " max " bp"
      print ">=1000 bp       : " c1000+0
      print ">=500 bp        : " c500+0
      print ">=200 bp        : " c200+0
    }'

# --- 过滤 >= 200 bp 送 BLAST ---
echo ""
echo "=== 过滤 >= 200 bp ==="
awk 'BEGIN{p=0}
  /^>/{header=$0; p=0; next}
  {seq=$0}
  length(seq) >= 200 {print header; print seq; p=1}' \
  ${CONTIGS} > ${TRANS}/contigs_200bp.fa

N_BLAST=$(grep -c "^>" ${TRANS}/contigs_200bp.fa 2>/dev/null || echo 0)
echo "${N_BLAST} contigs (>=200 bp) will be sent to BLAST"

# --- BLASTn remote ---
if [ "${N_BLAST}" -eq 0 ]; then
    echo "No contigs >= 200 bp. Check assembly quality or unmapped read count."
else
    echo ""
    echo "=== BLASTn (remote NCBI nt, may take several minutes) ==="
    conda run -n mag_biobakery \
    blastn \
      -query ${TRANS}/contigs_200bp.fa \
      -db nt \
      -remote \
      -outfmt "6 qseqid sseqid pident length qlen evalue bitscore stitle" \
      -max_target_seqs 5 \
      -evalue 1e-10 \
      -out ${TRANS}/blast_results.txt

    echo ""
    echo "=== BLAST Top Hits ==="
    printf "%-22s %6s %6s/%s  %10s  %s\n" "Contig" "Ident%" "Aln" "Len" "E-value" "Description"
    printf '%0.s-' {1..100}; echo
    sort -k7 -rn ${TRANS}/blast_results.txt | awk -F'\t' '!seen[$1]++' | head -20 | \
      awk -F'\t' '{printf "%-22s %5s%% %6s/%-6s %10s  %s\n", $1,$3,$4,$5,$6,substr($8,1,60)}'
fi

## 结果解读指南

### Task 1 — 物种/株系判断

| 全量比对率 | 解读 |
|-----------|------|
| ≥ 98% | 确认为中国仓鼠 (CHO) |
| 90–98% | 可能含少量外源污染 |
| < 90% | 非 CHO 或严重污染 |

| DHFR/侧翼深度比值 | 株系判断 |
|------------------|---------|
| < 0.05，且覆盖率 < 5% | DHFR 纯合缺失 → **CHO-DG44** |
| 0.05–0.35 | DHFR 半合缺失 → **CHO-DXB11** |
| > 0.70 | DHFR 完整 → **CHO-K1 或 CHO-S** |

### Task 2 — Transgene 结果解读

| BLAST 结果 | 可能解释 |
|-----------|---------|
| 无命中 | ① unmapped reads 均为测序噪音；② transgene 序列与 CHO 参考高度同源（已被比对捕获）|
| 命中 CMV / SV40 / polyA signal | 重组表达载体元件，确认有外源构建体 |
| 命中抗性基因（Puro / Neo / Hygro） | 选择标记，确认有外源 DNA 插入 |
| 命中人/小鼠基因 | 目的基因（GOI），需与客户确认 |
| 命中病毒载体骨架 | 可能残留包装载体片段 |
| Contigs 无命中但序列 ≥200 bp | 可能为客户保密序列，建议提供载体图谱做定向比对 |

### 局限性说明
- 本方法检测的是**游离/插入**的外源 DNA，**不能确定插入位置**（客户已知晓此点）
- 若 transgene 序列与 CHO 参考高度保守（如某些人源化基因），可能被宿主比对步骤"误吸"而无法检出 → 可改用更严格的参考比对（`-Y` flag + 较高 mapping quality 阈值）或直接向客户索取序列做定向 BLAST
- BLAST remote 速度依赖 NCBI 服务器负载，高峰期可能较慢；如 contigs 较多（>50条），建议分批提交或使用 NCBI Web BLAST 界面逐条确认